Let's handle our imports and load our dataset:

In [3]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import random 
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset

In [4]:
data = load_dataset("ag_news")

print(data["train"]["text"][0])

Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 2024046.63 examples/s]

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


We're defining the relevant training functions:


In [5]:
# tokenizer
def train_tokenizer(sentences, vocabulary_size=200):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocabulary_size)

    tokenizer.train_from_iterator(sentences, trainer)
    return tokenizer

In [6]:
# skipgrams
def make_skipgrams(sequence, vocabulary_size, window_size=3):
    couples = []
    labels = []

    for i in range(len(sequence)):
        target = sequence[i]
        left = max(0, i - window_size)
        right = min(len(sequence), i + window_size + 1)

        for j in range(left, right):
            if i != j:
                context = sequence[j]

                # positive pair
                couples.append((target, context))
                labels.append(1)

                # generate a random negative pair (in real life, you might want to generate several negative pairs per positive pair)
                negative_context = random.randint(1, vocabulary_size - 1)
                # TODO: we should probably check to make sure this isn't actually a positive pair
                couples.append((target, negative_context))
                labels.append(0)

    return [couples, labels]

def make_skipgrams_batch(tokenized_sentences, vocabulary_size, window_size=3):
    all_couples = []
    all_labels = []
    for tokenized_sentence in tokenized_sentences:
        couples, labels = make_skipgrams(tokenized_sentence, vocabulary_size, window_size)
        all_couples.extend(couples)
        all_labels.extend(labels)

    return [all_couples, all_labels]


In [17]:
# embedding model
def train_embedding_model(couples, labels, vocabulary_size):

    embedding_model = nn.Embedding(vocabulary_size, 50)
    skipgram_classifier = nn.Linear(2*50, 1) # target emb + context emb

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(list(embedding_model.parameters()) + list(skipgram_classifier.parameters()), lr=0.001) # concatenate the parameters for the embedding model and skipgram classifier

    pair_ids = torch.tensor(couples, dtype=torch.long)                # [N, 2]
    labels_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # [N, 1]

    for epoch in range(5000):
        target_ids = pair_ids[:, 0] # [N]
        context_ids = pair_ids[:, 1] # [N]

        target_embeddings = embedding_model(target_ids)  # [N, 50]
        context_embeddings = embedding_model(context_ids)  # [N, 50]

        target_context_together = torch.cat([target_embeddings, context_embeddings], dim=1)  # [N, 100]

        logits = skipgram_classifier(target_context_together)
        loss = loss_fn(logits, labels_tensor )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 500 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids[0] # use first subword token for this demo
    word_weights = embedding_model.weight.detach()
    return word_weights[word_ids]

In [21]:
# some back of the napkin math tells me this will take about 14 hours
# to run on my laptop with 5000 examples
sentences = data["train"]["text"][:10_000]

VOCAB_SIZE = 16000
tokenizer = train_tokenizer(sentences, VOCAB_SIZE)

# tokenized_sentences = [tokenizer.encode(sentence).ids for sentence in sentences]
tokenized_sentences = []
for example in sentences:
    ids = tokenizer.encode(example).ids
    tokens = tokenizer.encode(example).tokens
    tokenized_sentences.append(ids)

print("tokens:", tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, tokenizer.get_vocab_size())
embedding_model = train_embedding_model(couples, labels, tokenizer.get_vocab_size())




tokens: ['Feds', 'Bust', 'File', '-', 'Sharing', 'Sites', 'Operation', 'Digital', 'Gr', 'id', 'lock', 'targets', 'peer', '-', 'to', '-', 'peer', 'sites', ',', 'ISP', 's', 'in', 'piracy', 'investigation', '.']
epoch 500, loss=0.3938
epoch 1000, loss=0.3906
epoch 1500, loss=0.3889
epoch 2000, loss=0.3884
epoch 2500, loss=0.3882
epoch 3000, loss=0.3882
epoch 3500, loss=0.3881
epoch 4000, loss=0.3881
epoch 4500, loss=0.3881
epoch 5000, loss=0.3881


In [23]:
# checking aquire, buy, and baseball
acquire_embedding = get_embedding('acquire', tokenizer, embedding_model)
buy_embedding = get_embedding('buy', tokenizer, embedding_model)
investigation_embedding = get_embedding('investigation', tokenizer, embedding_model)

print("acquire-buy", F.cosine_similarity(acquire_embedding, buy_embedding, -1))
print("acquire-investigation", F.cosine_similarity(acquire_embedding, investigation_embedding, -1))


acquire-buy tensor(-0.2233)
acquire-investigation tensor(0.1656)


## Reflection
I tried a few times on up to 10,000 training samples, with a vocabulary of 16,000 tokens. Loss performance seemed like it plateaued relatively quickly - and still doesn't do very well. I tried fiddling with the learning rate as well, but it didn't seem to improve much at all. 

I suspect our neural network might not be big enough to adequately capture all the meaning in this large of a dataset, and using a bigger model means much more data. I tried comparing two similar business words (acquire and buy) and unrelated words (acquire and investigation), and didn't get much similarity or difference between them at all. 